# 🎯 Đánh Giá Suy Luận Mô Hình UAV-Anti-UAV ReID trên Kaggle GPU

Notebook này thiết lập kịch bản chuẩn hóa để kiểm tra lại toàn bộ các chỉ số đã được sử dụng để đánh giá mô hình **UAV-Anti-UAV ReID (DINOv3 ConvNeXt-Small + Bi-Mamba S6 + BNNeck)** trên tập dữ liệu benchmark **UAV-Anti-UAV**.

### 📌 Yêu cầu môi trường Kaggle:
- **Accelerator:** GPU T4 x2 hoặc P100 (16GB VRAM).
- **Internet:** **ON** (để tải cấu hình/tokenizer DINOv3 từ HuggingFace nếu cần).
- **Input Datasets:**
  1. `uav-anti-uav`: Bộ dữ liệu video gốc (chứa thư mục `Test/` hoặc cả `Train/` & `Test/`).
  2. `uav-reid-weights`: File trọng số đã huấn luyện `best_model.pth` (hoặc `best_model.pth.zip`).

In [ ]:
# [CELL 1] Kiểm tra GPU và Cài đặt thư viện phụ thuộc
!nvidia-smi
!pip install -q transformers timm packaging pyyaml opencv-python-headless matplotlib

In [ ]:
# [CELL 2] Thiết lập thư mục làm việc và Clone/Copy Code UAVAntiUAV
import os, sys, glob, shutil, zipfile, json
import torch
import yaml

WORKING_DIR = "/kaggle/working"
REPO_DIR = os.path.join(WORKING_DIR, "UAVAntiUAV")

if not os.path.exists(REPO_DIR):
    # Nếu đã clone hoặc tải mã nguồn lên input, copy sang working
    found_src = glob.glob("/kaggle/input/**/model.py", recursive=True)
    if found_src:
        src_dir = os.path.dirname(found_src[0])
        shutil.copytree(src_dir, REPO_DIR)
        print(f"-> Đã copy mã nguồn từ {src_dir} sang {REPO_DIR}")
    else:
        # Clone từ GitHub
        !git clone https://github.com/YolandCandy/UAVAntiUAV.git {REPO_DIR}

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Thư mục làm việc hiện tại:", os.getcwd())

In [ ]:
# [CELL 3] Tự động dò tìm đường dẫn Dataset và File trọng số (best_model.pth)
print("=== TÌM KIẾM DỮ LIỆU & TRỌNG SỐ TRÊN KAGGLE ===")

# 1. Tìm dataset Test của UAV-Anti-UAV
test_dirs = glob.glob("/kaggle/input/**/Test-002/Test", recursive=True)
if not test_dirs:
    test_dirs = glob.glob("/kaggle/input/**/Test", recursive=True)
if not test_dirs:
    test_dirs = glob.glob("/kaggle/input/**/UAV-Anti-UAV/Test", recursive=True)

if test_dirs:
    TEST_DATA_DIR = test_dirs[0]
    RAW_DATA_DIR = os.path.dirname(TEST_DATA_DIR)
    print(f"✅ Đã tìm thấy UAV-Anti-UAV Test Data tại: {TEST_DATA_DIR}")
else:
    raise FileNotFoundError("❌ Không tìm thấy thư mục Test của UAV-Anti-UAV trong /kaggle/input!")

# 2. Tìm file trọng số (pth hoặc zip)
pth_files = glob.glob("/kaggle/input/**/best_model.pth", recursive=True)
zip_files = glob.glob("/kaggle/input/**/best_model.pth.zip", recursive=True)

CHECKPOINT_PATH = "/kaggle/working/best_model.pth"
if pth_files:
    shutil.copy2(pth_files[0], CHECKPOINT_PATH)
    print(f"✅ Đã sao chép trọng số .pth từ: {pth_files[0]}")
elif zip_files:
    print(f"📦 Đang giải nén file zip trọng số từ: {zip_files[0]}...")
    with zipfile.ZipFile(zip_files[0], 'r') as zf:
        zf.extractall("/kaggle/working")
    # Kiểm tra sau giải nén
    extracted_pths = glob.glob("/kaggle/working/**/best_model.pth", recursive=True)
    if extracted_pths:
        CHECKPOINT_PATH = extracted_pths[0]
    print(f"✅ Đã giải nén checkpoint thành công: {CHECKPOINT_PATH}")
else:
    print("⚠️ Chưa tìm thấy file trọng số trong /kaggle/input, tìm kiếm trong /kaggle/working...")
    if os.path.exists(CHECKPOINT_PATH):
        print(f"✅ Sử dụng file có sẵn tại: {CHECKPOINT_PATH}")
    else:
        raise FileNotFoundError("❌ Không tìm thấy file best_model.pth!")

print(f"Dung lượng Checkpoint: {os.path.getsize(CHECKPOINT_PATH) / (1024*1024):.2f} MB")


In [ ]:
# [CELL 4] Kiểm tra tính hợp lệ và cấu trúc các layers của Model Checkpoint
from model import UAVReIDNet, load_checkpoint_verbose

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Khởi tạo UAVReIDNet (Backbone: dinov3_convnext) trên {device}...")

model = UAVReIDNet(backbone='dinov3_convnext')
load_summary = load_checkpoint_verbose(model, CHECKPOINT_PATH, tag="Kaggle_Check")
model.to(device)
model.eval()

if load_summary['backbone_missing']:
    print("❌ CẢNH BÁO: Backbone keys bị thiếu!")
else:
    print("✅ 100% Backbone và Bi-Mamba weights đã được nạp chính xác từ checkpoint!")

In [ ]:
# [CELL 5] Tạo File Cấu Hình config_kaggle_active.yaml
kaggle_config = {
    'device': {'platform': 'kaggle', 'gpu_jetson': False},
    'paths': {
        'raw_data_dir': RAW_DATA_DIR,
        'gasnet_dir': os.path.join(REPO_DIR, 'gasnet'),
        'data_dir': '/kaggle/working/processed',
        'checkpoint_dir': '/kaggle/working/checkpoints',
        'log_dir': '/kaggle/working/logs'
    },
    'data_pipeline': {
        'num_before_frames': 16,
        'num_after_frames': 16,
        'bbox_padding': 0.2,
        'crop_size': 256,
        'num_workers': 4,
        'frame_stride': 2
    },
    'train': {
        'num_frames': 12,  # ⚠️ QUAN TRỌNG: 12 frames để đồng bộ token
        'backbone': 'dinov3_convnext'
    },
    'eval': {
        'model_path': CHECKPOINT_PATH,
        'query_json': '/kaggle/working/processed/query_test.json',
        'gallery_json': '/kaggle/working/processed/gallery_test.json',
        'output_dir': '/kaggle/working/eval_results',
        'batch_size': 32,
        'num_workers': 4,
        'backbone': 'dinov3_convnext',
        'backbone_only': False,
        'reid_threshold': 0.75,
        'intra_sequence': False,
        'space': 'fused',
        'max_correct_vis': 0,
        'threshold_file': '/kaggle/working/calib_results/calibrated_threshold.json'
    },
    'calibration': {
        'output_dir': '/kaggle/working/calib_results',
        'cal_ratio': 0.4,
        'far_target': 0.001,
        'seed': 42
    },
    'infer': {
        'backbone': 'dinov3_convnext',
        'seq_dir': 'all',
        'test_dir': TEST_DATA_DIR,
        'model_path': CHECKPOINT_PATH,
        'output_video': '/kaggle/working/infer_output/output_reid.mp4',
        'out_dir': '/kaggle/working/infer_output',
        'stride': 2,
        'num_frames': 12,
        'bbox_padding': 0.2,
        'max_anchor_size': 5,
        'max_recent_size': 15,
        'soft_lock_threshold': 0.30,
        'reid_threshold': 0.75,
        'hijack_threshold': 0.40,
        'hijack_check_count': 5,
        'update_interval_sec': 2.0,
        'debug_sim': True
    },
    'infer_robustness': {
        'seq_dir': os.path.join(TEST_DATA_DIR, 'UAV-Anti-UAV_Test_000001'),
        'data_root': TEST_DATA_DIR,
        'model_path': CHECKPOINT_PATH,
        'output_video': '/kaggle/working/infer_output/output_robustness.mp4',
        'backbone': 'dinov3_convnext',
        'reid_threshold': 0.75,
        'num_frames': 12,
        'bbox_padding': 0.2,
        'num_imposters': 10
    }
}

CONFIG_FILE = "/kaggle/working/config_kaggle_active.yaml"
with open(CONFIG_FILE, 'w', encoding='utf-8') as f:
    yaml.dump(kaggle_config, f, default_flow_style=False)

print(f"✅ Đã tạo file cấu hình: {CONFIG_FILE}")


In [ ]:
# [CELL 6] Tiền Xử Lý Dữ Liệu (data_pipeline.py) - Tạo Query & Gallery Test
processed_dir = "/kaggle/working/processed"
query_test_path = os.path.join(processed_dir, "query_test.json")
gallery_test_path = os.path.join(processed_dir, "gallery_test.json")

if os.path.exists(query_test_path) and os.path.exists(gallery_test_path):
    print("✅ Đã có sẵn query_test.json và gallery_test.json. Bỏ qua bước trích xuất.")
else:
    print("🚀 Đang chạy data_pipeline.py trích xuất các cặp Query & Gallery từ Test sequences...")
    !python data_pipeline.py --config {CONFIG_FILE} --data-dir {RAW_DATA_DIR} --output-dir {processed_dir}

with open(query_test_path, 'r') as f:
    q_data = json.load(f)
print(f"Tổng số mẫu Query Test: {len(q_data)}")

In [ ]:
# [CELL 7] KỊCH BẢN 1A: Hiệu Chuẩn Ngưỡng Fixed-FAR 0.1% (calibrate_threshold.py)
print("🚀 Đang chạy calibrate_threshold.py (Fixed FAR <= 0.1%)...")
!python calibrate_threshold.py --config {CONFIG_FILE}

# Hiển thị các biểu đồ đã sinh ra
from IPython.display import Image, display
calib_dir = "/kaggle/working/calib_results"

plots = glob.glob(os.path.join(calib_dir, "*.png"))
for p in sorted(plots):
    print(f"📊 Biểu đồ: {os.path.basename(p)}")
    display(Image(filename=p, width=650))

In [ ]:
# [CELL 8] KỊCH BẢN 1B: Đánh Giá Global ReID (evaluate_reid.py) trên 2 không gian
print("🚀 [1/2] Đánh giá trên không gian FUSED (sau BatchNorm1d)...")
!python evaluate_reid.py --config {CONFIG_FILE} --space fused

print("\n🚀 [2/2] Đánh giá trên không gian PRE-BN (raw concat visual + temporal)...")
!python evaluate_reid.py --config {CONFIG_FILE} --space pre_bn

# Đọc và in bảng kết quả
eval_report_path = "/kaggle/working/eval_results/evaluation_report.json"
if os.path.exists(eval_report_path):
    with open(eval_report_path, 'r') as f:
        report = json.load(f)
    print("\n=== KẾT QUẢ ĐÁNH GIÁ GLOBAL REID ===")
    print(json.dumps(report, indent=2))

In [ ]:
# [CELL 9] KỊCH BẢN 2: Online Sequence Tracking & ReID Inference trên toàn bộ tập Test
# Chạy suy luận FSM trên toàn bộ các chuỗi sequence trong thư mục Test
print("🚀 Bắt đầu suy luận ReID trên toàn bộ tập Test (seq_dir: all)...")
!python infer.py --config {CONFIG_FILE} --seq-dir all


In [ ]:
# [CELL 10] KỊCH BẢN 3: Kiểm Tra Tính Kháng Nhiễu Imposter Attack (evaluate_reid_robustness.py)
print("🚀 Đang chạy bài kiểm tra kháng nhiễu đối thủ giả mạo (Imposters)...")
!python evaluate_reid_robustness.py --config {CONFIG_FILE}

# Hiển thị biểu đồ phân phối score giữa Genuine và Imposter
rob_plots = glob.glob("/kaggle/working/**/score_distribution*.png", recursive=True)
for p in rob_plots:
    print(f"📊 Biểu đồ Kháng Nhiễu: {p}")
    display(Image(filename=p, width=650))

In [ ]:
# [CELL 11] TỔNG HỢP VÀ ĐỐI CHIẾU CHỈ SỐ HOÀN CHỈNH
print("="*70)
print("              TỔNG HỢP TOÀN BỘ CHỈ SỐ MÔ HÌNH TRÊN KAGGLE")
print("="*70)

# 1. Đọc summary_metrics.txt
summary_txt = "/kaggle/working/infer_output/summary_metrics.txt"
if os.path.exists(summary_txt):
    with open(summary_txt, 'r') as f:
        print(f.read())

# 2. Đọc evaluation_report.json
if os.path.exists(eval_report_path):
    with open(eval_report_path, 'r') as f:
        e = json.load(f)
    print("--- GLOBAL REID COMPARISON TABLE ---")
    spaces = e.get('spaces', {})
    for sp, m in spaces.items():
        print(f"Không gian {sp.upper():<8}: Rank-1={m.get('rank1'):.2f}% | Rank-5={m.get('rank5'):.2f}% | mAP={m.get('map'):.2f}% | TAR@FAR={m.get('tar_at_far'):.2f}%")
print("="*70)